# Conformal drift detection & certified replay — GPU benchmarks (Colab)

Runs the full-scale **Overcooked** detection benchmark (the heavy *unmasked* JaxMARL env)
via the config-driven harness `code/scripts/run_benchmark.py`. The cheap envs (Simu, MPE
`simple_spread`) run on CPU and are produced outside this notebook.

> **SMAX is intentionally excluded from detection.** Its legal action set is
> state-dependent, which is incompatible with the benchmark's fixed-action partner-swap
> intervention (the harness rejects masked envs for `--experiment detection`). SMAX's
> validated contribution is the double-DQN next-state target-mask fix in
> `code/conformal_marl/iql.py`, covered by the training tests — not a detection run.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator: GPU` (a T4
is plenty for `cramped_room` / `asymm_advantages`).

Pipeline: check GPU -> clone repo -> install **JAX 0.4.30 + CUDA 12** (pinned; JaxMARL
0.0.2 needs `jax.tree_map`, removed in JAX 0.6) -> GPU sanity check -> run
`run_benchmark.py --full` for Overcooked (`cramped_room`, `asymm_advantages`) -> show & zip
figures.

See `code/RUN_ON_GPU.md` for the rationale and gotchas. This notebook is GPU-untested on
the author's machine (no local GPU) — treat the first run as the real verification.

## 0. Confirm a GPU is attached

In [ ]:
!nvidia-smi || echo 'No GPU visible — set Runtime -> Change runtime type -> GPU and re-run.'

## 1. Clone the repo

The repo is **private** (`ArnaudGardille/report-offline-marl-stability`). Provide a GitHub
token with read access when prompted (a fine-grained PAT scoped to this repo is safest).
The token is read into memory only and not written to disk. If you have already mounted the
repo another way, skip this cell and set `REPO_DIR` accordingly in the next cell.

In [ ]:
import getpass, os, subprocess

REPO = 'ArnaudGardille/report-offline-marl-stability'
REPO_DIR = '/content/report-offline-marl-stability'
BRANCH = 'main'  # or 'm8-standard-experiments' until merged

if not os.path.isdir(REPO_DIR):
    token = getpass.getpass('GitHub token (read access to the repo): ').strip()
    url = f'https://{token}@github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', url, REPO_DIR], check=True)
    del token, url
else:
    print('repo already present at', REPO_DIR)
print('contents:', os.listdir(REPO_DIR))

## 2. Install JAX 0.4.30 + CUDA 12 (pinned) and the project deps

**Do not bump JAX.** JaxMARL 0.0.2 calls `jax.tree_map` (removed in JAX 0.6), so we pin
`jax==jaxlib==0.4.30`. The `[cuda12_pip]` extra bundles CUDA 12 / cuDNN wheels (no system
CUDA toolkit needed). `chex` is capped (`<0.1.91` needs `jax>=0.7`). We `sys.path`-add
`code/` rather than `pip install -e .` because `pyproject.toml` pins Python `>=3.12,<3.13`
while Colab may run 3.10/3.11 (the code runs fine on 3.10+, only the metadata pin is strict).

In [ ]:
# JAX 0.4.30 with CUDA 12 (the -f index is required to find the +cuda12 jaxlib wheel).
!pip install -q --upgrade "jax[cuda12_pip]==0.4.30" jaxlib==0.4.30 \
    -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
# Project deps at their pinned versions (chex capped to keep a tree_map-bearing jax).
!pip install -q "jaxmarl==0.0.2" "chex<0.1.91" flax optax flashbax distrax matplotlib
print('\nInstall done. If the GPU check below shows CPU, do Runtime -> Restart session, '
      'then re-run from THIS cell (the new jaxlib needs a fresh interpreter).')

## 3. GPU sanity check

`jax.devices()` should list a `cuda`/`gpu` device and the matmul should run on it. If it
still shows CPU, **Runtime -> Restart session** and re-run from the install cell.

In [ ]:
import jax, jax.numpy as jnp
print('jax', jax.__version__, '| backend', jax.default_backend())
print('devices:', jax.devices())
x = jnp.ones((2048, 2048))
y = jax.block_until_ready(x @ x)  # runs on the default device
print('matmul placed on:', list(y.devices()))
assert jax.default_backend() == 'gpu', 'GPU not active -- restart the runtime and re-run the install cell.'

## 4. Point Python at the repo's `code/` package

In [ ]:
import sys, os
CODE_DIR = os.path.join(REPO_DIR, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
# Smoke import to confirm the package + JaxMARL resolve under this jax.
from conformal_marl.envs.registry import make_env, available_envs
print('factory envs:', available_envs())
print('cwd:', os.getcwd())

## 5. (Optional) quick smoke on GPU

Runs the tiny `--smoke` Overcooked detection config on the GPU first (CI-time) to confirm
the whole pipeline executes end-to-end before launching the long `--full` runs.

In [ ]:
!python scripts/run_benchmark.py --env overcooked --experiment detection --smoke --layout cramped_room

## 6. Full Overcooked detection (camera-ready)

Anytime-valid conformal martingale vs AES at matched false-alarm, on Overcooked
`cramped_room` and `asymm_advantages`. `--full` uses 8 seeds, longer pre/post windows, and
the env's tuned IQL budget (see the `SCALE` table in `scripts/run_benchmark.py`); the
multi-seed sweep is what the GPU accelerates. The two layouts write to distinct files
(`m8_overcooked_<layout>_detection_full.*`). Lower the budgets in `SCALE` for a short session.

In [ ]:
!python scripts/run_benchmark.py --env overcooked --experiment detection --full --layout cramped_room

In [ ]:
!python scripts/run_benchmark.py --env overcooked --experiment detection --full --layout asymm_advantages

## 7. A note on SMAX (excluded from detection)

SMAX is **not** run as a detection benchmark here. Detection's fixed-action partner-swap
intervention assumes a *state-independent* action set, which SMAX violates (units die /
come into range, so the legal action set changes with state) — the harness rejects masked
envs for `--experiment detection`. SMAX's validated contribution in this project is the
**double-DQN next-state target-mask fix** in `code/conformal_marl/iql.py`. The next cell
exercises that fix's unit test.

In [ ]:
# SMAX's contribution is the next-state target-mask fix (not a detection run); run its unit test.
!python -m pytest tests/test_iql_learns.py -k next_state_mask -q

## 8. (Optional) selection ablations on the heavy envs

`uniform` vs `conformal_policy` / `exact_ratio` replay selection. On easily-learned envs
selection tends to be neutral-to-slower (see `scripts/ablation.py`); detection is the
headline. Uncomment to run.

In [ ]:
# !python scripts/run_benchmark.py --env overcooked --experiment selection --full --layout cramped_room
# (selection is optional/illustrative; on easily-learned envs it is neutral-to-slower. SMAX
#  selection would train via the mask-correct IQL loop but uses a generic budget, so it is
#  not featured here.)

## 9. Show & download the figures

In [ ]:
import glob
from IPython.display import Image, display
figs = sorted(glob.glob(os.path.join(CODE_DIR, 'results', 'm8_*_full.png')))
print('figures:', [os.path.basename(f) for f in figs])
for f in figs:
    display(Image(filename=f))

In [ ]:
# Zip the full-scale figures + raw json for download from the Colab file browser.
import shutil, glob, os
out = '/content/m8_gpu_figures'
os.makedirs(out, exist_ok=True)
for f in glob.glob(os.path.join(CODE_DIR, 'results', 'm8_*_full.png')):
    shutil.copy(f, out)
for f in glob.glob(os.path.join(CODE_DIR, 'outputs', 'm8_*_full.json')):
    shutil.copy(f, out)
shutil.make_archive('/content/m8_gpu_figures', 'zip', out)
print('wrote /content/m8_gpu_figures.zip')
try:
    from google.colab import files
    files.download('/content/m8_gpu_figures.zip')
except Exception as e:
    print('download via the file browser; auto-download skipped:', e)